In [0]:
dbutils.widgets.text("catalog","dev")
catalog = dbutils.widgets.get("catalog")
schema = "bikeshare"
volume = "raw"
VOL = f"/Volumes/{catalog}/{schema}/{volume}"
print(VOL)

In [0]:
import requests

DISCOVERY_URL = "https://sharedmobility.ch/gbfs.json"   # GBFS 2.0, no key needed

r = requests.get(DISCOVERY_URL, timeout = 30)
r.raise_for_status()
disc = r.json()

lang = next(iter(disc["data"]))
feeds = {f["name"]: f["url"] for f in disc["data"][lang]["feeds"] }
feeds

In [0]:
import json
from datetime import datetime, timezone
from zoneinfo import ZoneInfo

NEEDED_DATA = ["station_status", "station_information"]

ts = datetime.now(timezone.utc).strftime("%Y-%m-%d-%H-%M-%SZ")

print(f"Writing to {VOL}")
print(f"Timestamp: {ts}")

for name in NEEDED_DATA:
    payload = requests.get(feeds[name], timeout=30).json()
    folder = f"{VOL}/{name}"
    dbutils.fs.mkdirs(folder)
    with open (f"{folder}/{name}_{ts}.json", "w") as fh:
        json.dump(payload, fh)
    print(name, len(payload["data"]["stations"]), "stations ->", folder)

In [0]:
%sql
SELECT 
    _metadata.file_name,
    from_utc_timestamp(TIMESTAMP(last_updated), "Europe/Zurich") as last_updated_ts,
    SIZE(data.stations) AS n_stations
FROM read_files(
    concat('/Volumes/', :catalog, '/bikeshare/raw/station_status/'),
    format => 'json');